# NurseBot
**Voice-controlled robot nurse simulation** — Doctor & patient commands · Real-time map · Task queue with interrupt

Run all cells top to bottom, then open:
- `http://localhost:7861` — hospital map (robot moves here)
- `http://localhost:7860` — control panel (Gradio)

See `README.md` for full usage and `docs/` for architecture notes.

## Setup

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in [
    "faster-whisper", "kokoro>=0.9.4", "sounddevice",
    "soundfile", "numpy", "scipy", "requests", "flask", "gradio>=4.16",
]:
    install(pkg)

print("done")

### Ollama (optional)
Enables smarter intent parsing via `llama3.2`. Skip if you don't have it — the rule parser works fine.

```bash
brew install ollama
ollama serve          # keep running in a terminal
ollama pull llama3.2
```

In [ ]:
import requests as _req

def _ollama_up(model="llama3.2"):
    try:
        r = _req.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200 and any(model in m["name"] for m in r.json().get("models", []))
    except Exception:
        return False

OLLAMA_AVAILABLE = _ollama_up()
print("Ollama: ready" if OLLAMA_AVAILABLE else "Ollama: not found — rule parser will be used")

## Imports & configuration

In [ ]:
import io, json, re, socket, threading, time, uuid
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum, IntEnum
from typing import Callable, Dict, List, Optional

import numpy as np
import scipy.signal
import sounddevice as sd
import soundfile as sf
import requests
from faster_whisper import WhisperModel
from kokoro import KPipeline
from IPython.display import Audio, display

STT_SR = 16_000
TTS_SR = 24_000

WHISPER_MODEL = "base.en"
KOKORO_VOICE  = "af_heart"
OLLAMA_MODEL  = "llama3.2"
OLLAMA_URL    = "http://localhost:11434/api/generate"

FLASK_PORT  = 7861
GRADIO_PORT = 7860

ROLE_PERMS = {
    "doctor":  {"can_interrupt": True, "can_delete_any": True,  "can_set_urgent": True},
    "patient": {"can_interrupt": True, "can_delete_any": False, "can_set_urgent": False},
}

TASK_LIBRARY = {
    "blood pressure": {"dur": 12, "steps": ["Prepare cuff", "Measure BP", "Record"]},
    "check blood":    {"dur": 12, "steps": ["Prepare cuff", "Measure BP", "Record"]},
    "medication":     {"dur": 18, "steps": ["Verify Rx", "Prepare dose", "Confirm ID", "Administer", "Log"]},
    "medicine":       {"dur": 15, "steps": ["Verify Rx", "Prepare dose", "Administer", "Log"]},
    "temperature":    {"dur":  6, "steps": ["Prepare thermometer", "Measure", "Record"]},
    "draw blood":     {"dur": 20, "steps": ["Tourniquet", "Draw sample", "Label", "Send to lab"]},
    "iv":             {"dur": 10, "steps": ["Prepare bag", "Verify", "Replace", "Check flow"]},
    "dressing":       {"dur": 15, "steps": ["Gather supplies", "Remove old", "Clean wound", "Apply new"]},
    "wound":          {"dur": 15, "steps": ["Gather supplies", "Remove old", "Clean wound", "Apply new"]},
    "assessment":     {"dur": 25, "steps": ["Check consciousness", "Assess pain", "Vitals", "Document"]},
    "vital":          {"dur":  8, "steps": ["Check BP", "Check pulse", "Check O2", "Record"]},
    "ecg":            {"dur": 12, "steps": ["Apply electrodes", "Run ECG", "Print results"]},
    "emergency":      {"dur": 10, "steps": ["Assess patient", "Call physician", "Stabilise", "Document"]},
}

HOSPITAL_ROOMS = {
    "medicine": {"label": "Medicine Cabinet"},
    "lab":      {"label": "Laboratory"},
    "icu":      {"label": "ICU"},
    "301":      {"label": "Room 301"},
    "302":      {"label": "Room 302"},
    "303":      {"label": "Room 303"},
    "304":      {"label": "Room 304"},
    "305":      {"label": "Room 305"},
    "306":      {"label": "Room 306"},
    "station":  {"label": "Nurse Station"},
}

def task_info(name):
    nl = name.lower()
    for k, v in TASK_LIBRARY.items():
        if k in nl:
            return v
    return {"dur": 10, "steps": ["Prepare", "Execute", "Finalise"]}

print("config loaded")

## Data models

In [ ]:
class Priority(IntEnum):
    LOW    = 1
    NORMAL = 2
    URGENT = 3

class TaskStatus(Enum):
    QUEUED      = "queued"
    RUNNING     = "running"
    INTERRUPTED = "interrupted"
    COMPLETED   = "completed"
    DELETED     = "deleted"

@dataclass
class Task:
    id:           str        = field(default_factory=lambda: uuid.uuid4().hex[:8])
    name:         str        = "Unnamed Task"
    description:  str        = ""
    priority:     Priority   = Priority.NORMAL
    created_by:   str        = "doctor"
    status:       TaskStatus = TaskStatus.QUEUED
    progress:     float      = 0.0
    created_at:   datetime   = field(default_factory=datetime.now)
    started_at:   Optional[datetime] = None
    completed_at: Optional[datetime] = None
    estimated_duration: float = 10.0

## Robot controller

In [ ]:
class RobotController:
    def __init__(self):
        self.current   = "station"
        self.target    = "station"
        self.status    = "idle"
        self.action    = "Standing by"
        self.task_name = ""
        self.progress  = 0.0
        self.carrying  = None
        self._lock     = threading.Lock()

    def _dest(self, name):
        t = name.lower()
        m = re.search(r"room\s*(\d+)", t)
        if m and m.group(1) in HOSPITAL_ROOMS:
            return m.group(1)
        if any(w in t for w in ["medicine", "medication", "administer"]):
            return "medicine"
        if any(w in t for w in ["lab", "blood sample", "draw blood"]):
            return "lab"
        if any(w in t for w in ["icu", "critical", "intensive"]):
            return "icu"
        return "station"

    def assign_task(self, task):
        with self._lock:
            loc = self._dest(task.name)
            self.target    = loc
            self.task_name = task.name
            self.carrying  = "medicine" if loc == "medicine" else "sample" if loc == "lab" else "equipment"
            self.status    = "urgent" if task.priority == Priority.URGENT else "moving"
            self.action    = f"Moving to {HOSPITAL_ROOMS[loc]['label']}"

    def arrive(self):
        with self._lock:
            self.current = self.target
            self.status  = "working"
            self.action  = f"Working at {HOSPITAL_ROOMS[self.current]['label']}"

    def update_progress(self, p, step):
        with self._lock:
            self.progress = p
            lbl = HOSPITAL_ROOMS.get(self.current, {}).get("label", "")
            self.action = f"{step} — {lbl}"

    def complete(self):
        with self._lock:
            self.target   = "station"
            self.status   = "returning"
            self.action   = "Returning to nurse station"
            self.carrying = None
            self.progress = 1.0

    def idle(self):
        with self._lock:
            self.current   = "station"
            self.status    = "idle"
            self.action    = "Standing by"
            self.task_name = ""
            self.progress  = 0.0
            self.carrying  = None

    def get_state(self):
        with self._lock:
            return {
                "current":  self.current,
                "target":   self.target,
                "status":   self.status,
                "action":   self.action,
                "task":     self.task_name,
                "progress": round(self.progress, 3),
                "carrying": self.carrying,
            }

## TTS — Kokoro

In [ ]:
class TTSEngine:
    def __init__(self, voice=KOKORO_VOICE):
        print(f"Loading Kokoro ({voice})…")
        self.pipeline = KPipeline(lang_code="a")
        self.voice    = voice
        self._lock    = threading.Lock()
        print("TTS ready.")

    def synth(self, text):
        chunks = [a for _, _, a in self.pipeline(text, voice=self.voice, speed=0.93)]
        return np.concatenate(chunks) if chunks else np.zeros(100, dtype=np.float32)

    def speak(self, text, blocking=True):
        with self._lock:
            audio = self.synth(text)
            display(Audio(audio, rate=TTS_SR, autoplay=True))
            sd.play(audio, samplerate=TTS_SR)
            if blocking:
                sd.wait()

    def speak_async(self, text):
        threading.Thread(target=self.speak, args=(text,), daemon=True).start()

## STT — Faster-Whisper

In [ ]:
class STTEngine:
    def __init__(self, model_size=WHISPER_MODEL):
        print(f"Loading Faster-Whisper ({model_size})…")
        self.model = WhisperModel(model_size, device="cpu", compute_type="int8")
        print("STT ready.")

    def transcribe(self, audio, samplerate=STT_SR):
        if audio is None or len(audio) == 0:
            return ""
        if samplerate != STT_SR:
            n     = int(len(audio) * STT_SR / samplerate)
            audio = scipy.signal.resample(audio, n).astype(np.float32)
        audio = audio.astype(np.float32)
        if audio.max() > 1.0:
            audio /= 32768.0
        buf = io.BytesIO()
        sf.write(buf, audio, STT_SR, format="WAV")
        buf.seek(0)
        segs, _ = self.model.transcribe(buf, beam_size=5, language="en")
        return " ".join(s.text.strip() for s in segs).strip()

## Intent parser

In [ ]:
_SYSTEM_PROMPT = """Parse the voice command. Return ONLY compact JSON, no markdown.
Keys: action, task_name, task_description, priority, target_task_name, response_to_user, confidence
action: add_task | interrupt_now | delete_task | list_tasks | task_status | unknown
priority: urgent | normal | low
Only doctors may set urgent. Patient pain/help → interrupt_now urgent.
response_to_user: 1-2 warm professional sentences the robot speaks aloud.""""

class RuleParser:
    _URGENT   = {"urgent","emergency","pain","chest","help","bleeding","severe","falling","acute"}
    _DELETE   = {"cancel","delete","remove","abort"}
    _LIST     = {"list","show","status","what","tasks","queue","running"}
    _TASK_MAP = {
        "blood pressure":"Check Blood Pressure", "bp":"Check Blood Pressure",
        "medication":"Administer Medication",     "medicine":"Administer Medication",
        "temperature":"Take Temperature",         "vital":"Vital Signs Check",
        "draw blood":"Draw Blood Sample",         "iv":"Change IV",
        "drip":"Change IV",                       "dressing":"Wound Dressing",
        "wound":"Wound Dressing",                 "assessment":"Patient Assessment",
        "oxygen":"Check Oxygen Level",            "ecg":"ECG Recording",
        "emergency":"Emergency Response",
    }

    def _priority(self, t, role):
        if any(w in t for w in self._URGENT): return "urgent"
        if any(w in t for w in {"low","whenever","later","routine"}): return "low"
        return "normal"

    def _name(self, text):
        t = text.lower()
        for kw, name in self._TASK_MAP.items():
            if kw in t:
                r = re.search(r"room\s*(\d+)", t)
                return f"{name} — Room {r.group(1)}" if r else name
        result = t
        for f in ["please","can you","i need","nurse","add","check on","the","a ","urgent","emergency","now"]:
            result = result.replace(f, " ")
        return re.sub(r"\s+", " ", result).strip().title()[:50] or "General Task"

    def parse(self, text, role):
        t   = text.lower().strip()
        pri = self._priority(t, role)

        if role == "patient" and any(w in t for w in {"pain","help","emergency","chest","bleeding"}):
            return {"action":"interrupt_now","task_name":"Emergency Patient Request",
                    "task_description":text,"priority":"urgent","target_task_name":None,
                    "response_to_user":"I hear you. Flagging as emergency — help is coming right away.",
                    "confidence":0.93}

        if any(w in t for w in self._DELETE):
            n = self._name(text)
            return {"action":"delete_task","task_name":n,"task_description":text,"priority":"normal",
                    "target_task_name":text,"response_to_user":f"Cancelling: {n}.","confidence":0.85}

        if any(w in t for w in self._LIST) and "blood" not in t:
            return {"action":"list_tasks","task_name":"","task_description":"","priority":"normal",
                    "target_task_name":None,"response_to_user":"Here is the current task board.",
                    "confidence":0.90}

        if role == "doctor" and pri == "urgent":
            n = self._name(text)
            return {"action":"interrupt_now","task_name":n,"task_description":text,"priority":"urgent",
                    "target_task_name":None,"response_to_user":f"Urgent acknowledged. Moving for: {n}.",
                    "confidence":0.88}

        n = self._name(text)
        return {"action":"add_task","task_name":n or "General Task","task_description":text,
                "priority":pri,"target_task_name":None,
                "response_to_user":f"Adding task: {n}. Priority: {pri}.","confidence":0.80}


class IntentParser:
    def __init__(self, model=OLLAMA_MODEL):
        self.model  = model
        self._rule  = RuleParser()
        self.use_ai = OLLAMA_AVAILABLE
        mode = f"Ollama ({model})" if self.use_ai else "rule parser"
        print(f"Intent parser: {mode}")

    def parse(self, text, role):
        if not self.use_ai:
            return self._rule.parse(text, role)
        try:
            resp = requests.post(
                OLLAMA_URL,
                json={"model": self.model,
                      "prompt": f"{_SYSTEM_PROMPT}\n\nROLE: {role.upper()}\nCOMMAND: {text}",
                      "stream": False},
                timeout=12,
            )
            raw = resp.json()["response"].strip()
            s, e = raw.find("{"), raw.rfind("}") + 1
            return json.loads(raw[s:e]) if s >= 0 and e > s else self._rule.parse(text, role)
        except Exception:
            return self._rule.parse(text, role)

## Task manager

In [ ]:
class TaskManager:
    def __init__(self, robot):
        self.robot = robot
        self.tasks: Dict[str, Task] = {}
        self._queue: List[Task] = []
        self.current_task: Optional[Task] = None
        self.interrupted_stack: List[Task] = []
        self._log: List[str] = []
        self._lock          = threading.RLock()
        self._stop_evt      = threading.Event()
        self._interrupt_evt = threading.Event()
        self._thread        = None
        self.tts            = None

    def _say(self, msg):
        ts = datetime.now().strftime("%H:%M:%S")
        self._log.append(f"[{ts}] {msg}")
        if len(self._log) > 100:
            self._log.pop(0)
        print(f"  Nurse: {msg}")
        if self.tts:
            self.tts.speak_async(msg)

    def add_task(self, name, description, priority, created_by):
        with self._lock:
            t = Task(name=name, description=description, priority=priority,
                     created_by=created_by, estimated_duration=task_info(name)["dur"])
            self.tasks[t.id] = t
            self._queue.append(t)
            self._queue.sort(key=lambda x: (-x.priority.value, x.created_at))
            self._say(f"Task added: '{name}'. Priority: {priority.name.lower()}.")
            return t

    def interrupt_with(self, urgent):
        with self._lock:
            if self.current_task and self.current_task.status == TaskStatus.RUNNING:
                self.current_task.status = TaskStatus.INTERRUPTED
                self.interrupted_stack.append(self.current_task)
                self._interrupt_evt.set()
                self._say(f"Pausing '{self.current_task.name}'. Urgent: '{urgent.name}'.")
            else:
                self._say(f"Executing urgent: '{urgent.name}'.")
            self._queue.insert(0, urgent)

    def delete_task(self, identifier, role):
        with self._lock:
            target = next(
                (t for t in self.tasks.values()
                 if identifier.lower() in t.name.lower() or t.id == identifier),
                None,
            )
            if not target:
                return False, f"No task matching '{identifier}'."
            if role == "patient" and target.created_by == "doctor":
                return False, "Patients cannot delete doctor-assigned tasks."
            if target.status in (TaskStatus.COMPLETED, TaskStatus.DELETED):
                return False, f"Task already {target.status.value}."

            if target.status == TaskStatus.RUNNING:
                target.status = TaskStatus.DELETED
                self._interrupt_evt.set()
                msg = f"Cancelling running task: '{target.name}'."
            elif target.status == TaskStatus.INTERRUPTED:
                target.status = TaskStatus.DELETED
                self.interrupted_stack = [t for t in self.interrupted_stack if t.id != target.id]
                msg = f"Deleted paused task: '{target.name}'."
            else:
                target.status = TaskStatus.DELETED
                self._queue = [t for t in self._queue if t.id != target.id]
                msg = f"Deleted queued task: '{target.name}'."

            self._say(msg)
            return True, msg

    def status_report(self):
        with self._lock:
            parts = []
            if self.current_task:
                parts.append(f"Running '{self.current_task.name}' at {int(self.current_task.progress * 100)}%.")
            q = [t for t in self._queue if t.status == TaskStatus.QUEUED]
            if q:
                parts.append("Queued: " + ", ".join(t.name for t in q[:4]) + ".")
            if self.interrupted_stack:
                parts.append("Paused: " + ", ".join(t.name for t in self.interrupted_stack) + ".")
            return " ".join(parts) if parts else "Queue is empty."

    def _run_task(self, task):
        self.robot.assign_task(task)
        with self._lock:
            task.status     = TaskStatus.RUNNING
            task.started_at = datetime.now()
            task.progress   = 0.0
            self.current_task = task
            self._interrupt_evt.clear()

        info  = task_info(task.name)
        steps = info["steps"]
        dur   = info["dur"]

        time.sleep(min(1.8, dur * 0.15))
        if not self._interrupt_evt.is_set() and task.status != TaskStatus.DELETED:
            self.robot.arrive()

        for i, step in enumerate(steps):
            if self._interrupt_evt.is_set() or self._stop_evt.is_set():
                break
            self.robot.update_progress((i + 0.5) / len(steps), step)
            print(f"  [{task.name}] {step}")
            for j in range(10):
                if self._interrupt_evt.is_set() or self._stop_evt.is_set():
                    break
                time.sleep(dur / len(steps) / 10)
                p = (i * 10 + j + 1) / (len(steps) * 10)
                task.progress = p
                self.robot.update_progress(p, step)

        with self._lock:
            interrupted = self._interrupt_evt.is_set()
            self._interrupt_evt.clear()

            if task.status != TaskStatus.DELETED and not interrupted:
                task.status       = TaskStatus.COMPLETED
                task.completed_at = datetime.now()
                task.progress     = 1.0
                self._say(f"Completed: '{task.name}'.")
                self.robot.complete()
                time.sleep(1.5)
                self.robot.idle()

                if self.interrupted_stack:
                    resume = self.interrupted_stack.pop()
                    resume.status   = TaskStatus.QUEUED
                    resume.progress = 0.0
                    self._queue.insert(0, resume)
                    self._say(f"Resuming: '{resume.name}'.")
            elif task.status == TaskStatus.DELETED:
                self.robot.complete()
                time.sleep(0.8)
                self.robot.idle()

            self.current_task = None

    def _worker(self):
        while not self._stop_evt.is_set():
            with self._lock:
                nxt = self._queue.pop(0) if self._queue and self.current_task is None else None
            if nxt:
                self._run_task(nxt)
            else:
                time.sleep(0.3)

    def start(self):
        self._stop_evt.clear()
        self._thread = threading.Thread(target=self._worker, name="NurseWorker", daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_evt.set()
        if self._thread:
            self._thread.join(timeout=3)

## Nurse system

In [ ]:
class NurseSystem:
    def __init__(self, tm, tts, stt, parser):
        self.tm     = tm
        self.tts    = tts
        self.stt    = stt
        self.parser = parser
        tm.tts = tts

    def process(self, text, role):
        if not text or not text.strip():
            return "No command received."

        print(f"\n[{role.upper()}] {text}")
        intent   = self.parser.parse(text, role)
        action   = intent.get("action", "unknown")
        response = intent.get("response_to_user", "Command received.")
        perms    = ROLE_PERMS[role]

        if action == "add_task":
            p_str = intent.get("priority", "normal")
            if p_str == "urgent" and not perms["can_set_urgent"]:
                p_str = "normal"
            p_map    = {"urgent": Priority.URGENT, "normal": Priority.NORMAL, "low": Priority.LOW}
            priority = p_map.get(p_str, Priority.NORMAL)
            name     = intent.get("task_name", "Unspecified Task")

            if priority == Priority.URGENT:
                t = Task(name=name, description=intent.get("task_description", ""),
                         priority=priority, created_by=role,
                         estimated_duration=task_info(name)["dur"])
                self.tm.tasks[t.id] = t
                self.tm.interrupt_with(t)
            else:
                self.tm.add_task(name, intent.get("task_description", ""), priority, role)

        elif action == "interrupt_now":
            if not perms["can_interrupt"]:
                response = "You do not have permission to interrupt."
            else:
                name = intent.get("task_name") or "Emergency Request"
                t = Task(name=name, description=intent.get("task_description", text),
                         priority=Priority.URGENT, created_by=role,
                         estimated_duration=task_info(name)["dur"])
                self.tm.tasks[t.id] = t
                self.tm.interrupt_with(t)

        elif action == "delete_task":
            target = intent.get("target_task_name") or intent.get("task_name", "")
            ok, msg = self.tm.delete_task(target, role)
            if not ok:
                response = msg

        elif action in ("list_tasks", "task_status"):
            response = self.tm.status_report()

        print(f"  Nurse: {response}")
        return response

    def voice_round(self, role, duration=5):
        text = self.stt.transcribe(
            sd.rec(int(duration * STT_SR), samplerate=STT_SR, channels=1, dtype="float32")
        )
        sd.wait()
        print(f'Heard: "{text}"')
        return self.process(text, role) if text else "Nothing heard."

## Flask map server

Serves the hospital map at `localhost:7861`. The map's JS calls `/state` every 300ms and moves the robot.
This runs independently from Gradio — no Svelte reactivity involved.

In [ ]:
from flask import Flask, Response, jsonify
import logging

logging.getLogger("werkzeug").setLevel(logging.ERROR)

_MAP_HTML = """<!DOCTYPE html><html><head><meta charset="UTF-8"><title>NurseBot Map</title>
<style>
*{box-sizing:border-box;margin:0;padding:0}
body{background:#0f172a;overflow:hidden;font-family:-apple-system,sans-serif}
#hmap{position:relative;width:100vw;height:100vh;background:#1a2035}
.ch{position:absolute;background:#243050}
.cv{position:absolute;background:#243050}
.hr{position:absolute;border-radius:10px;display:flex;flex-direction:column;
    align-items:center;justify-content:center;font-weight:600;color:white;
    border:2px solid rgba(255,255,255,0.18);transition:box-shadow .35s,transform .25s}
.hr.active{box-shadow:0 0 0 3px #fff,0 0 26px 6px rgba(255,210,0,.75);transform:scale(1.05);z-index:6}
.ri{font-size:22px;margin-bottom:2px}
.rl{font-size:10px;opacity:.88;text-align:center;line-height:1.2}
#rw{position:absolute;pointer-events:none;z-index:10}
#bub{position:absolute;background:rgba(255,255,255,.95);color:#1a2035;border-radius:8px;
     padding:5px 10px;font-size:11px;font-weight:600;bottom:calc(100% + 8px);left:50%;
     transform:translateX(-50%);max-width:160px;text-align:center;line-height:1.3;
     box-shadow:0 2px 8px rgba(0,0,0,.3)}
#bub::after{content:"";position:absolute;top:100%;left:50%;transform:translateX(-50%);
            border:5px solid transparent;border-top-color:rgba(255,255,255,.95)}
#cbadge{position:absolute;top:-10px;right:-10px;font-size:16px}
</style></head><body>
<div id="hmap">
<div class="ch" style="left:0;top:31%;width:100%;height:5%"></div>
<div class="ch" style="left:0;top:54%;width:100%;height:5%"></div>
<div class="ch" style="left:0;top:77%;width:100%;height:5%"></div>
<div class="cv" style="left:31%;top:0;width:5%;height:100%"></div>
<div class="cv" style="left:63%;top:0;width:5%;height:100%"></div>
<div class="hr" id="hr-medicine" style="left:2%;top:2%;width:27%;height:27%;background:#4c1d95">
  <div class="ri">&#128138;</div><div class="rl">Medicine Cabinet</div></div>
<div class="hr" id="hr-lab" style="left:36%;top:2%;width:27%;height:27%;background:#064e3b">
  <div class="ri">&#129514;</div><div class="rl">Laboratory</div></div>
<div class="hr" id="hr-icu" style="left:68%;top:2%;width:30%;height:27%;background:#7f1d1d">
  <div class="ri">&#128680;</div><div class="rl">ICU</div></div>
<div class="hr" id="hr-301" style="left:2%;top:37%;width:27%;height:16%;background:#1e3a5f">
  <div class="ri">&#128716;</div><div class="rl">Room 301</div></div>
<div class="hr" id="hr-302" style="left:36%;top:37%;width:27%;height:16%;background:#1e3a5f">
  <div class="ri">&#128716;</div><div class="rl">Room 302</div></div>
<div class="hr" id="hr-303" style="left:68%;top:37%;width:30%;height:16%;background:#1e3a5f">
  <div class="ri">&#128716;</div><div class="rl">Room 303</div></div>
<div class="hr" id="hr-304" style="left:2%;top:60%;width:27%;height:16%;background:#1e3a5f">
  <div class="ri">&#128716;</div><div class="rl">Room 304</div></div>
<div class="hr" id="hr-305" style="left:36%;top:60%;width:27%;height:16%;background:#1e3a5f">
  <div class="ri">&#128716;</div><div class="rl">Room 305</div></div>
<div class="hr" id="hr-306" style="left:68%;top:60%;width:30%;height:16%;background:#1e3a5f">
  <div class="ri">&#128716;</div><div class="rl">Room 306</div></div>
<div class="hr" id="hr-station" style="left:25%;top:83%;width:50%;height:14%;
     background:#0c4a6e;flex-direction:row;gap:8px;font-size:13px">
  <span style="font-size:18px">&#127968;</span><span>Nurse Station</span></div>
<div id="rw" style="left:48%;top:82%">
  <div id="bub">Standing by</div>
  <span id="cbadge"></span>
  <svg id="rsvg" width="54" height="74" viewBox="0 0 54 74">
    <ellipse cx="27" cy="70" rx="17" ry="5" fill="rgba(0,0,0,.22)"/>
    <rect x="11" y="57" width="13" height="11" rx="5.5" fill="#334155" id="rwl"/>
    <rect x="30" y="57" width="13" height="11" rx="5.5" fill="#334155" id="rwr"/>
    <rect x="10" y="22" width="34" height="38" rx="10" fill="#3b82f6" id="rbody"/>
    <rect x="22" y="27" width="10" height="26" rx="3" fill="white" opacity=".88"/>
    <rect x="14" y="34" width="26" height="8" rx="3" fill="white" opacity=".88"/>
    <rect x="2" y="24" width="9" height="23" rx="4" fill="#2563eb" id="rarmL"/>
    <rect x="43" y="24" width="9" height="23" rx="4" fill="#2563eb" id="rarmR"/>
    <rect x="21" y="14" width="12" height="10" rx="2" fill="#334155"/>
    <rect x="9" y="1" width="36" height="28" rx="10" fill="#3b82f6" id="rhead"/>
    <rect x="13" y="5" width="28" height="17" rx="6" fill="#0f172a" opacity=".8"/>
    <rect x="15" y="8" width="10" height="10" rx="3" fill="#22d3ee" id="reL"/>
    <rect x="29" y="8" width="10" height="10" rx="3" fill="#22d3ee" id="reR"/>
    <circle cx="20" cy="13" r="3" fill="#0e7490"/>
    <circle cx="34" cy="13" r="3" fill="#0e7490"/>
    <line x1="27" y1="1" x2="27" y2="-9" stroke="#3b82f6" stroke-width="2.5"/>
    <circle cx="27" cy="-12" r="5" fill="#f97316" id="rant"/>
  </svg>
</div>
</div>
<script>
var POS = {
  medicine:{x:16,y:15}, lab:{x:50,y:15}, icu:{x:83,y:15},
  "301":{x:16,y:45}, "302":{x:50,y:45}, "303":{x:83,y:45},
  "304":{x:16,y:68}, "305":{x:50,y:68}, "306":{x:83,y:68},
  station:{x:50,y:90}
};
var EYES = {idle:"#22d3ee",moving:"#a3e635",working:"#facc15",urgent:"#f87171",returning:"#34d399"};
var BODY = {idle:"#3b82f6",moving:"#2563eb",working:"#16a34a",urgent:"#dc2626",returning:"#0891b2"};
var CARRY = {medicine:"&#128138;",equipment:"&#128296;",sample:"&#129514;"};
var rx=50, ry=90, fr=0, state={status:"idle",target:"station",action:"Standing by",carrying:null};

function lerp(a,b,t){return a+(b-a)*t}
function toCanvas(px,py){return{x:px/100*window.innerWidth,y:py/100*window.innerHeight}}

function recolor(s){
  var ec=EYES[s]||EYES.idle, bc=BODY[s]||BODY.idle;
  ["reL","reR"].forEach(function(id){var e=document.getElementById(id);if(e)e.setAttribute("fill",ec)});
  ["rbody","rhead","rarmL","rarmR"].forEach(function(id){var e=document.getElementById(id);if(e)e.setAttribute("fill",bc)});
  var a=document.getElementById("rant");
  if(a) a.setAttribute("fill", s==="urgent"?"#ef4444":"#f97316");
}

function tick(){
  fr++;
  var w = document.getElementById("rw");
  if(!w){requestAnimationFrame(tick);return}
  var tp = toCanvas((POS[state.target]||POS.station).x, (POS[state.target]||POS.station).y);
  rx = lerp(rx, tp.x, 0.07);
  ry = lerp(ry, tp.y, 0.07);
  w.style.left = (rx-27)+"px";
  w.style.top  = (ry-37)+"px";

  var moving = Math.abs(rx-tp.x)>3 || Math.abs(ry-tp.y)>3;
  var svg = document.getElementById("rsvg");
  if(svg){
    if(moving){
      var bob  = Math.sin(fr*0.38)*3;
      var lean = Math.sign(tp.x-rx)*Math.min(Math.abs(tp.x-rx)*0.04,9);
      svg.style.transform = "translateY("+bob+"px) rotate("+lean+"deg)";
    } else {
      svg.style.transform = "translateY("+Math.sin(fr*0.04)*2+"px)";
      if(state.status==="working"){
        var sw = Math.sin(fr*0.18)*14;
        var al = document.getElementById("rarmL"), ar = document.getElementById("rarmR");
        if(al) al.style.transform = "rotate("+sw+"deg)";
        if(ar) ar.style.transform = "rotate("+(-sw)+"deg)";
      }
    }
  }
  recolor(state.status);
  requestAnimationFrame(tick);
}

function poll(){
  fetch("/state")
    .then(function(r){return r.json()})
    .then(function(s){
      state = s;
      var b = document.getElementById("bub");
      if(b) b.textContent = s.action || "Standing by";
      var cb = document.getElementById("cbadge");
      if(cb) cb.textContent = s.carrying ? CARRY[s.carrying]||"" : "";
      document.querySelectorAll(".hr").forEach(function(r){r.classList.remove("active")});
      if(s.target && s.target!=="station" && (s.status==="working"||s.status==="moving"||s.status==="urgent")){
        var el = document.getElementById("hr-"+s.target);
        if(el) el.classList.add("active");
      }
    })
    .catch(function(){})
    .finally(function(){setTimeout(poll,300)});
}

requestAnimationFrame(tick);
poll();
</script></body></html>"""

def _port_free(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", p)) != 0

def start_flask(robot, port=FLASK_PORT):
    app = Flask(__name__)

    @app.after_request
    def cors(r):
        r.headers["Access-Control-Allow-Origin"] = "*"
        return r

    @app.route("/state")
    def state():
        return jsonify(robot.get_state())

    @app.route("/")
    def map_page():
        return Response(_MAP_HTML, mimetype="text/html")

    t = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port, debug=False,
                                use_reloader=False, threaded=True),
        daemon=True,
    )
    t.start()
    return app

## Gradio UI

In [ ]:
import gradio as gr

def _board(tm):
    parts = []
    with tm._lock:
        if tm.current_task:
            t = tm.current_task
            pct = int(t.progress * 100)
            icon = {"URGENT":"🔴","NORMAL":"🟡","LOW":"🟢"}.get(t.priority.name, "⚪")
            parts.append(
                f'<div style="background:#e8f4fd;border-left:4px solid #1565c0;border-radius:8px;padding:10px;margin-bottom:8px">'
                f'<div style="display:flex;justify-content:space-between"><b>&#9654; {t.name}</b>'
                f'<span style="background:#1565c0;color:#fff;font-size:10px;padding:2px 8px;border-radius:99px">RUNNING</span></div>'
                f'<div style="font-size:11px;color:#555;margin:3px 0">{icon} by {t.created_by}</div>'
                f'<div style="background:#bbdefb;height:8px;border-radius:4px;overflow:hidden">'
                f'<div style="background:#1565c0;width:{pct}%;height:100%;border-radius:4px;transition:width .4s"></div></div>'
                f'<div style="font-size:12px;color:#1565c0;font-weight:600;margin-top:3px">{pct}%</div></div>'
            )

        for t in reversed(tm.interrupted_stack):
            parts.append(
                f'<div style="background:#fff8e1;border-left:4px solid #f57f17;border-radius:8px;padding:9px;margin-bottom:6px">'
                f'<div style="display:flex;justify-content:space-between"><b>&#9208; {t.name}</b>'
                f'<span style="background:#f57f17;color:#fff;font-size:10px;padding:2px 8px;border-radius:99px">PAUSED</span></div>'
                f'<div style="font-size:11px;color:#666">{int(t.progress*100)}% saved</div></div>'
            )

        for t in tm._queue:
            bg, bc, lbl = (
                ("#ffebee","#c62828","URGENT") if t.priority==Priority.URGENT else
                ("#fffde7","#f9a825","NORMAL") if t.priority==Priority.NORMAL else
                ("#e8f5e9","#388e3c","LOW")
            )
            parts.append(
                f'<div style="background:{bg};border-left:4px solid {bc};border-radius:8px;padding:8px;margin-bottom:5px">'
                f'<div style="display:flex;justify-content:space-between">'
                f'<span style="font-size:12px;font-weight:500">{t.name}</span>'
                f'<span style="background:{bc};color:#fff;font-size:10px;padding:1px 7px;border-radius:99px">{lbl}</span></div>'
                f'<div style="font-size:11px;color:#666">by {t.created_by}</div></div>'
            )

        done = [t for t in tm.tasks.values() if t.status==TaskStatus.COMPLETED]
        for t in done[-3:]:
            parts.append(
                f'<div style="opacity:.6;background:#f1f8e9;border-left:3px solid #81c784;'
                f'border-radius:8px;padding:6px;margin-bottom:4px;font-size:11px">'
                f'&#x2705; <b>{t.name}</b> &middot; by {t.created_by}</div>'
            )

        if not tm.current_task and not tm._queue and not tm.interrupted_stack:
            parts.append('<div style="text-align:center;color:#aaa;padding:24px;font-size:13px">&#x1F4A4; Idle</div>')

    return '<div style="font-family:-apple-system,sans-serif">' + "".join(parts) + "</div>"


def build_ui(nurse, tts, stt):
    tm = nurse.tm

    def get_log():
        return "\n".join(reversed(list(tm._log)[-18:])) if tm._log else "No activity yet."

    def send_text(text, role):
        if not text or not text.strip():
            return "", _board(tm), get_log(), None, "Type a command"
        resp  = nurse.process(text.strip(), role.lower())
        audio = tts.synth(resp)
        return "", _board(tm), get_log(), (TTS_SR, audio), f"Sent: {text.strip()}"

    def send_voice(audio_data, role):
        if audio_data is None:
            return _board(tm), get_log(), None, "No audio"
        try:
            sr, data = audio_data
            if data.ndim > 1: data = data.mean(axis=1)
            data = data.astype(np.float32)
            if data.max() > 1.0: data /= 32768.0
            text = stt.transcribe(data, samplerate=sr)
            if not text:
                return _board(tm), get_log(), None, "Could not hear — speak louder"
            resp     = nurse.process(text, role.lower())
            audio_out = tts.synth(resp)
            return _board(tm), get_log(), (TTS_SR, audio_out), f'Heard: "{text}"'
        except Exception as e:
            return _board(tm), get_log(), None, f"Error: {e}"

    def refresh():
        return _board(tm), get_log()

    css = (
        ".big{min-height:52px!important;font-size:14px!important;font-weight:600!important}"
        ".emg{min-height:58px!important;font-size:16px!important;font-weight:700!important}"
    )

    with gr.Blocks(title="NurseBot", theme=gr.themes.Soft(primary_hue="blue"), css=css) as demo:

        gr.HTML(
            f'<div style="text-align:center;padding:12px 0 8px;border-bottom:1px solid #e5e7eb;margin-bottom:12px">'
            f'<h1 style="font-size:22px;font-weight:700;margin:0">&#x1F3E5; NurseBot</h1>'
            f'<p style="color:#6b7280;font-size:12px;margin:4px 0 0">'
            f'Map: <a href="http://localhost:{FLASK_PORT}/" target="_blank" style="color:#3b82f6">'
            f'localhost:{FLASK_PORT}</a></p></div>'
        )

        with gr.Row(equal_height=False):
            with gr.Column(scale=6):
                gr.HTML(
                    f'<iframe src="http://localhost:{FLASK_PORT}/" width="100%" height="455"'
                    f' style="border:none;border-radius:12px;overflow:hidden;background:#1a2035"></iframe>'
                )
                gr.HTML('<p style="font-weight:600;font-size:14px;margin:10px 0 6px">Task queue</p>')
                board = gr.HTML(value=_board(tm))

            with gr.Column(scale=3, min_width=280):
                role = gr.Radio(choices=["Doctor","Patient"], value="Doctor", label="Role",
                                info="Doctor — full control · Patient — requests and emergency only")

                gr.HTML('<hr style="margin:10px 0;border-color:#e5e7eb">')
                gr.HTML('<p style="font-weight:600;font-size:13px;margin:0 0 4px">Text command</p>')
                text_in  = gr.Textbox(
                    placeholder="check blood pressure room 302 · urgent room 301 · cancel medication",
                    label="", lines=2)
                send_btn = gr.Button("Send", variant="primary", elem_classes=["big"])

                gr.HTML('<hr style="margin:10px 0;border-color:#e5e7eb">')
                gr.HTML(
                    '<p style="font-weight:600;font-size:13px;margin:0 0 2px">Voice command</p>'
                    '<p style="font-size:11px;color:#9ca3af;margin:0 0 6px">Auto-processes when you stop recording</p>'
                )
                voice_in  = gr.Audio(sources=["microphone"], type="numpy", label="", show_label=False)
                heard_box = gr.Textbox(label="Status", lines=1, interactive=False,
                                       value="Record, speak, then stop")

                gr.HTML('<hr style="margin:10px 0;border-color:#e5e7eb">')
                gr.HTML('<p style="font-weight:600;font-size:13px;margin:0 0 6px">Nurse response</p>')
                audio_out = gr.Audio(label="", autoplay=True, interactive=False, show_label=False)

                gr.HTML('<hr style="margin:10px 0;border-color:#e5e7eb">')
                gr.HTML('<p style="font-weight:600;font-size:13px;margin:0 0 6px">Quick actions</p>')
                with gr.Row():
                    qbp = gr.Button("Check BP",    variant="secondary")
                    qmd = gr.Button("Medication",  variant="secondary")
                    qtp = gr.Button("Temperature", variant="secondary")
                with gr.Row():
                    qvt = gr.Button("Vitals",      variant="secondary")
                    qas = gr.Button("Assessment",  variant="secondary")
                    qst = gr.Button("Status",      variant="secondary")
                qhelp = gr.Button("🆘  EMERGENCY", variant="stop", elem_classes=["emg"])

                gr.HTML('<hr style="margin:10px 0;border-color:#e5e7eb">')
                log_box = gr.Textbox(value="", label="Activity log", lines=8,
                                     interactive=False, max_lines=12)

        timer = gr.Timer(value=0.5)
        timer.tick(fn=refresh, outputs=[board, log_box])

        txt_outs = [text_in, board, log_box, audio_out, heard_box]
        send_btn.click(fn=send_text, inputs=[text_in, role], outputs=txt_outs)
        text_in.submit(fn=send_text, inputs=[text_in, role], outputs=txt_outs)
        voice_in.change(fn=send_voice, inputs=[voice_in, role],
                        outputs=[board, log_box, audio_out, heard_box])

        for btn, cmd in [
            (qbp, "check blood pressure"), (qmd, "administer medication"),
            (qtp, "take temperature"),     (qvt, "vital signs check"),
            (qas, "patient assessment"),   (qst, "task status"),
            (qhelp, "emergency I need help now"),
        ]:
            btn.click(fn=lambda r, c=cmd: send_text(c, r), inputs=[role], outputs=txt_outs)

    return demo

## Launch

Run this cell last. Two browser windows open:
- **Map** → `http://localhost:7861` — robot moves here when tasks run
- **Controls** → `http://localhost:7860` — send commands here

In [ ]:
import webbrowser

robot  = RobotController()
tts    = TTSEngine()
stt    = STTEngine()
parser = IntentParser()
tm     = TaskManager(robot)
nurse  = NurseSystem(tm, tts, stt, parser)

tm.start()

if _port_free(FLASK_PORT):
    start_flask(robot, FLASK_PORT)
    time.sleep(1.0)

tts.speak("NurseBot online. Ready for commands.")

webbrowser.open(f"http://localhost:{FLASK_PORT}/")

demo = build_ui(nurse, tts, stt)
demo.launch(inbrowser=True, server_port=GRADIO_PORT, share=False, quiet=True)